In [2]:
!pip install -q evaluate rouge-score bert-score sacrebleu sentencepiece
!pip install -q -U "bitsandbytes>=0.46.1" accelerate

In [3]:
import json
import time
import torch
import evaluate
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from bert_score import score

DATASET_PATH = "/kaggle/input/datasets/ibadurrahmanmemon/omnimind/omnimind_finetune_dataset.jsonl"
dataset = []

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        dataset.append(json.loads(line))

TEST_SIZE = 100
evaluation_set = dataset[:TEST_SIZE]

prompts, references = [], []
for sample in evaluation_set:
    for m in sample["messages"]:
        if m["role"] == "user": prompts.append(m["content"])
        if m["role"] == "assistant": references.append(m["content"])

print(f"Loaded {len(prompts)} test samples.")

Loaded 100 test samples.


In [4]:
BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer_base = AutoTokenizer.from_pretrained(BASE_MODEL)
model_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map={"": 0}
)
print("Base model loaded in 4-bit!")

config.json:   0%|          | 0.00/956 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Base model loaded in 4-bit!


In [5]:
def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id # Prevents console warnings
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

base_predictions, base_times = [], []

for prompt in tqdm(prompts, desc="Generating Base Responses"):
    start = time.time()
    response = generate_response(model_base, tokenizer_base, prompt)
    base_times.append(time.time() - start)
    base_predictions.append(response)

Generating Base Responses: 100%|██████████| 100/100 [15:26<00:00,  9.27s/it]


In [6]:
# Load Evaluators
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")

# Compute Scores
base_rouge = rouge.compute(predictions=base_predictions, references=references)
base_bleu = bleu.compute(predictions=base_predictions, references=[[x] for x in references])
base_meteor = meteor.compute(predictions=base_predictions, references=references)
_, _, F1 = score(base_predictions, references, lang="en")
base_bert = F1.mean().item()

# Save to a DataFrame and export to CSV for Notebook 2
base_results = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "METEOR", "BERTScore", "Latency"],
    "Base": [
        base_rouge["rouge1"], base_rouge["rouge2"], base_rouge["rougeL"], 
        base_bleu["bleu"], base_meteor["meteor"], base_bert, 
        sum(base_times)/len(base_times)
    ]
})

base_results.to_csv("base_metrics.csv", index=False)
print("\nBase evaluation complete! File 'base_metrics.csv' saved.")

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Base evaluation complete! File 'base_metrics.csv' saved.
